# FINAL CONTROLLED LLM STUDY
## Frozen MedGemma vs new QLoRA × No RAG / Pre-rerank RAG / Post-rerank RAG × Run-1 / Run-2 decoding

This notebook is deliberately **LLM-only**. It does **not** recompute BioMedCLIP embeddings, FAISS retrieval, quadrant features, or XGBoost reranking.

### Primary 12-condition design

| Generator | Context | Run-1 | Run-2 |
|---|---|---:|---:|
| Frozen MedGemma | No retrieval | ✓ | ✓ |
| Frozen MedGemma | Pre-rerank RAG | ✓ | ✓ |
| Frozen MedGemma | Post-rerank RAG | ✓ | ✓ |
| New QLoRA | No retrieval | ✓ | ✓ |
| New QLoRA | Pre-rerank RAG | ✓ | ✓ |
| New QLoRA | Post-rerank RAG | ✓ | ✓ |

**Run-1** preserves the original inference behaviour (`max_new_tokens=500`, deterministic, `repetition_penalty=1.15`) but uses the corrected parser/evaluator. A parse/generation failure is a technical failure — it is never converted into five `Not determined` fields.

**Run-2** reproduces the later robustness corrections: repetition penalty 1.20, 6-gram blocking, correct MedGemma termination, first-complete-valid-JSON stopping, concise output constraint, and 350-token ceiling.



In [1]:
# ============================================================
# 0. INSTALL — TIMED
# ============================================================
import time
_t0=time.time()
!pip -q install -U "transformers>=4.53" "bitsandbytes>=0.46.1" peft trl accelerate datasets evaluate rouge_score nltk scikit-learn pandas openpyxl
print(f"[TIME] Installation: {(time.time()-_t0)/60:.2f} min")


  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 62.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 992.6/992.6 kB 70.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 50.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 99.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 157.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 147.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 53.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.5 wh

In [2]:
# ============================================================
# 1. IMPORTS + PATHS
# ============================================================
import os, re, ast, json, math, gc, random, time
from contextlib import contextmanager
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig,
    TrainerCallback, EarlyStoppingCallback,
    StoppingCriteria, StoppingCriteriaList,
)
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from trl.trainer.sft_trainer import DataCollatorForLanguageModeling

SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# ===================== EDIT THESE FOUR PATHS =====================
RAG_BEFORE_PATH="/content/rag_output_exp2c_hubfixed.json"
RAG_AFTER_PATH ="/content/rag_output_exp2d_final_enriched.json"
FULL_PATIENT_PATH="/content/mmdental_cleaned_full.csv"
DEV_POOL_PATH="/content/train_exp1_combined.csv"
# ================================================================

OUTPUT_DIR="/kaggle/working/final_12_condition_llm_study"
os.makedirs(OUTPUT_DIR,exist_ok=True)

BASE_MODEL="unsloth/medgemma-1.5-4b-it"
TARGET_FIELDS=["Oral Check","Diagnosis","Treatment plan","Handle","Doctor advices"]
REFERENCE_FIELDS=[
    "Main appeal","Present medical history","Past medical history","Oral Check","Diagnosis",
    "Treatment plan","Handle","Doctor advices"
]

@contextmanager
def timed(label):
    t=time.time()
    print(f"\n[TIMER START] {label}")
    try: yield
    finally:
        sec=time.time()-t
        print(f"[TIME] {label}: {sec:.1f}s ({sec/60:.2f} min)")

def fmt_time(sec):
    sec=max(0,int(sec)); h,r=divmod(sec,3600); m,s=divmod(r,60)
    return f"{h}h {m}m {s}s" if h else (f"{m}m {s}s" if m else f"{s}s")

print("CUDA:",torch.cuda.is_available())
if torch.cuda.is_available(): print("GPU:",torch.cuda.get_device_name(0))


CUDA: True
GPU: NVIDIA A100-SXM4-80GB


## 2. Load and validate the frozen pre-/post-rerank test artifacts


In [3]:
with timed("Load + validate test artifacts"):
    def load_json(p):
        with open(p,encoding="utf-8") as f:return json.load(f)

    def norm_id(x):
        if x is None:return None
        try:return str(int(float(x)))
        except:return str(x).strip()

    def normalize_rag(d):
        if isinstance(d,list):
            d={norm_id(x.get("case_id",x.get("query_id",i))):x for i,x in enumerate(d)}
        out={}
        for k,v in d.items():
            q=norm_id(v.get("case_id",v.get("query_id",k)))
            ex=v.get("retrieved_exemplars",v.get("retrieved_cases",v.get("exemplars",[]))) or []
            out[q]={**v,"case_id":q,"retrieved_exemplars":ex}
        return out

    before=normalize_rag(load_json(RAG_BEFORE_PATH))
    after =normalize_rag(load_json(RAG_AFTER_PATH))
    assert set(before)==set(after)
    TEST_IDS=list(after.keys())
    assert len(TEST_IDS)==101 and len(set(TEST_IDS))==101

    def ex_id(ex):return norm_id(ex.get("case_id",ex.get("patient_id",ex.get("id"))))

    for label,rag in [("BEFORE",before),("AFTER",after)]:
        assert min(len(rag[q]["retrieved_exemplars"]) for q in TEST_IDS)>=5
        selfhits=sum(ex_id(e)==q for q in TEST_IDS for e in rag[q]["retrieved_exemplars"][:5])
        assert selfhits==0
        print(label,"queries:",len(rag),"self hits:",selfhits)

    if "88" in after:
        ids=[ex_id(x) for x in after["88"]["retrieved_exemplars"][:5]]
        print("Query 88 post-rerank fingerprint:",ids)
        print("Expected final fingerprint: ['150','10','57','386','60']")



[TIMER START] Load + validate test artifacts
BEFORE queries: 101 self hits: 0
AFTER queries: 101 self hits: 0
Query 88 post-rerank fingerprint: ['150', '10', '57', '386', '60']
Expected final fingerprint: ['150','10','57','386','60']
[TIME] Load + validate test artifacts: 0.0s (0.00 min)


## 3. Load clinical tables and reconstruct the leakage-controlled 276 → 248/28 split


In [4]:
with timed("Clinical data + leakage audit + 248/28 split"):
    def read_table(p):
        if p.lower().endswith((".xlsx",".xls")):return pd.read_excel(p)
        try:return pd.read_csv(p)
        except:return pd.read_excel(p)

    def id_col(df):
        for c in ["Filename","case_id","patient_id","Patient ID","ID"]:
            if c in df.columns:return c
        raise KeyError("No patient ID column")

    def clean(v,default=""):
        if v is None:return default
        try:
            if pd.isna(v):return default
        except:pass
        s=str(v).strip()
        return default if s.lower() in {"","nan","none","null","n/a","na"} else s

    full_df=read_table(FULL_PATIENT_PATH); dev_df=read_table(DEV_POOL_PATH)
    FID=id_col(full_df); DID=id_col(dev_df)
    full_df[FID]=full_df[FID].map(norm_id); dev_df[DID]=dev_df[DID].map(norm_id)
    full={norm_id(r[FID]):r for _,r in full_df.iterrows()}
    dev ={norm_id(r[DID]):r for _,r in dev_df.iterrows()}

    assert len(dev)==320, f"Expected original 320 development patients, got {len(dev)}"
    testset=set(TEST_IDS)
    overlap=set(dev)&testset
    assert len(overlap)==44, f"Expected 44 overlapping IDs, got {len(overlap)}"

    safe_order=[norm_id(x) for x in dev_df[DID].tolist() if norm_id(x) not in testset]
    assert len(safe_order)==276
    train_ids,val_ids=train_test_split(safe_order,test_size=0.1,random_state=42)
    assert len(train_ids)==248 and len(val_ids)==28
    assert set(train_ids).isdisjoint(testset) and set(val_ids).isdisjoint(testset)

    print("Initial development:",320)
    print("Removed identity overlaps:",len(overlap))
    print("Safe development:",len(safe_order))
    print("Train:",len(train_ids),"Validation:",len(val_ids),"Test:",len(TEST_IDS))



[TIMER START] Clinical data + leakage audit + 248/28 split
Initial development: 320
Removed identity overlaps: 44
Safe development: 276
Train: 248 Validation: 28 Test: 101
[TIME] Clinical data + leakage audit + 248/28 split: 0.7s (0.01 min)


## 4. Build new leak-free QLoRA train/validation JSONL — no embeddings


In [5]:
with timed("Build query-safe TF-IDF SFT references + JSONL"):
    def safe_text(row):
        return (
            f"Age: {clean(row.get('Age'),'Unknown')}. "
            f"Sex: {clean(row.get('Sex'),'Unknown')}. "
            f"Main appeal: {clean(row.get('Main appeal'),'Unknown')}"
        )

    train_text=[safe_text(dev[x]) for x in train_ids]
    vectorizer=TfidfVectorizer(lowercase=True,ngram_range=(1,2),sublinear_tf=True)
    X=vectorizer.fit_transform(train_text)
    train_pos={x:i for i,x in enumerate(train_ids)}

    def get_safe_refs(qid,k=5):
        qv=vectorizer.transform([safe_text(dev[qid])])
        sims=cosine_similarity(qv,X)[0]
        order=np.argsort(-sims)
        out=[]
        for j in order:
            cid=train_ids[j]
            if cid==qid:continue
            out.append((cid,float(sims[j])))
            if len(out)==k:return out
        return out

    train_refs={q:get_safe_refs(q) for q in train_ids}
    val_refs={q:get_safe_refs(q) for q in val_ids}

    SYSTEM_PROMPT=(
        "You are a dental clinical documentation assistant. You will be given a patient's "
        "Age, Sex, and Main appeal (chief complaint), along with similar PAST patient cases "
        "retrieved for reference.\n\n"
        "CRITICAL: Retrieved cases are REFERENCE MATERIAL from OTHER patients. They are NOT "
        "facts about the current patient. Do not assume the current patient has the same findings, "
        "tooth numbers, diagnoses, or treatments unless the current patient's own information "
        "genuinely supports it.\n\n"
        "Rules:\n"
        "1. NEVER invent or copy a tooth number, ICD code, diagnosis, finding, medication, "
        "procedure, or treatment that is not directly supported by the current patient's own "
        "Main appeal or by a clear, justified pattern across the retrieved cases.\n"
        "2. If there is insufficient evidence to determine a field, output exactly "
        "\"Not determined\" for that field. Do not guess.\n"
        "3. Output ONLY one valid JSON object with exactly these 5 keys, in this order: "
        "Oral Check, Diagnosis, Treatment plan, Handle, Doctor advices."
    )

    def hist_record(cid):
        r=full.get(cid,dev.get(cid))
        return {f:clean(r.get(f),"Not available") for f in REFERENCE_FIELDS}

    def user_prompt_from_refs(qid,refs):
        r=dev[qid]
        current=(
            "CURRENT PATIENT:\n"
            f"Age: {clean(r.get('Age'),'Unknown')}\n"
            f"Sex: {clean(r.get('Sex'),'Unknown')}\n"
            f"Main appeal: {clean(r.get('Main appeal'),'Unknown')}\n\n"
            "REFERENCE CASES — OTHER PATIENTS; not facts about the current patient:\n"
        )
        blocks=[]
        for i,(rid,_) in enumerate(refs,1):
            rr=hist_record(rid)
            body="\n".join(f"{f}: {rr[f]}" for f in REFERENCE_FIELDS)
            blocks.append(f"[Reference Case {i}]\n{body}")
        return current+"\n\n".join(blocks)

    def target(qid):
        r=dev[qid]
        return {f:clean(r.get(f),"Not determined") for f in TARGET_FIELDS}

    def examples(ids,refs):
        return [{
            "case_id":q,
            "messages":[
                {"role":"system","content":SYSTEM_PROMPT},
                {"role":"user","content":user_prompt_from_refs(q,refs[q])},
                {"role":"assistant","content":json.dumps(target(q),ensure_ascii=False)}
            ]
        } for q in ids]

    train_rows=examples(train_ids,train_refs); val_rows=examples(val_ids,val_refs)
    TRAIN_JSONL=os.path.join(OUTPUT_DIR,"train_248_new_qlora.jsonl")
    VAL_JSONL=os.path.join(OUTPUT_DIR,"val_28_new_qlora.jsonl")
    for p,rows in [(TRAIN_JSONL,train_rows),(VAL_JSONL,val_rows)]:
        with open(p,"w",encoding="utf-8") as f:
            for x in rows:f.write(json.dumps(x,ensure_ascii=False)+"\n")
    print("Saved:",TRAIN_JSONL,len(train_rows))
    print("Saved:",VAL_JSONL,len(val_rows))



[TIMER START] Build query-safe TF-IDF SFT references + JSONL
Saved: /kaggle/working/final_12_condition_llm_study/train_248_new_qlora.jsonl 248
Saved: /kaggle/working/final_12_condition_llm_study/val_28_new_qlora.jsonl 28
[TIME] Build query-safe TF-IDF SFT references + JSONL: 0.5s (0.01 min)


## 5. Load MedGemma once for frozen experiments


In [6]:
with timed("Load frozen MedGemma"):
    bnb=BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    tokenizer=AutoTokenizer.from_pretrained(BASE_MODEL)
    if tokenizer.pad_token is None:tokenizer.pad_token=tokenizer.eos_token
    frozen=AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,device_map="auto",quantization_config=bnb,
        torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
        attn_implementation="sdpa",
    )
    frozen.eval(); frozen.config.use_cache=True



[TIMER START] Load frozen MedGemma


config.json:   0%|          | 0.00/2.59k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 33.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

... (5 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/174 [00:00<?, ?B/s]

[TIME] Load frozen MedGemma: 33.7s (0.56 min)


## 6. Common test prompt + corrected JSON handling


In [7]:
def exemplar_record(ex):
    rec=ex.get("record",ex.get("clinical_record"))
    if isinstance(rec,dict):return rec
    cid=ex_id(ex)
    assert cid in full,f"Missing retrieved record {cid}"
    return full[cid]

def query_row(qid):
    assert qid in full,f"Missing test patient {qid}"
    return full[qid]

def build_test_user(qid,rag=None,k=0):
    r=query_row(qid)
    current=(
        "CURRENT PATIENT:\n"
        f"Age: {clean(r.get('Age'),'Unknown')}\n"
        f"Sex: {clean(r.get('Sex'),'Unknown')}\n"
        f"Main appeal: {clean(r.get('Main appeal'),'Unknown')}"
    )
    if rag is None or k==0:
        return current+"\n\nNo retrieved historical cases are provided."
    blocks=[]
    for i,ex in enumerate(rag[qid]["retrieved_exemplars"][:k],1):
        rr=exemplar_record(ex)
        body="\n".join(f"{f}: {clean(rr.get(f),'Not available')}" for f in REFERENCE_FIELDS)
        blocks.append(f"[Reference Case {i} — OTHER PATIENT]\n{body}")
    return current+"\n\nREFERENCE CASES — OTHER PATIENTS; not facts about the current patient:\n"+"\n\n".join(blocks)

EXPECTED_FIELDS=TARGET_FIELDS.copy()

def find_first_complete_json(text):
    if not text:return None,None
    for start in [i for i,c in enumerate(text) if c=="{"]:
        depth=0; ins=False; esc=False
        for i in range(start,len(text)):
            ch=text[i]
            if esc:esc=False;continue
            if ch=="\\" and ins:esc=True;continue
            if ch=='"':ins=not ins;continue
            if ins:continue
            if ch=="{":depth+=1
            elif ch=="}":
                depth-=1
                if depth==0:
                    candidate=text[start:i+1]
                    try:obj=json.loads(candidate)
                    except:break
                    if isinstance(obj,dict) and set(obj.keys())==set(EXPECTED_FIELDS):
                        return {f:str(obj[f]).strip() for f in EXPECTED_FIELDS},candidate
                    break
    return None,None

OUTPUT_CONSTRAINT="""
OUTPUT CONSTRAINT:
Return exactly one JSON object containing exactly these five keys:
"Oral Check", "Diagnosis", "Treatment plan", "Handle", "Doctor advices".
Keep every field concise.
Use at most 60 words for "Oral Check".
Use at most 20 words for "Diagnosis".
Use at most 30 words for "Treatment plan".
Use at most 80 words for "Handle".
Use at most 30 words for "Doctor advices".
Do not create lists of additional findings.
Do not repeat procedures.
Do not generate another JSON object.
End immediately after the final closing brace.
""".strip()

class FirstValidJSONStoppingCriteria(StoppingCriteria):
    def __init__(self,tok,prompt_len):
        self.tok=tok;self.prompt_len=prompt_len
    def __call__(self,input_ids,scores,**kwargs):
        raw=self.tok.decode(input_ids[0][self.prompt_len:],skip_special_tokens=False)
        obj,_=find_first_complete_json(raw)
        return obj is not None


## 7. Run-1 and Run-2 generation functions

**Important:** both use the same corrected technical-success definition and the same later evaluator. The only intentional difference is decoding/inference behaviour.


In [8]:
@torch.inference_mode()
def generate_case(model,tok,qid,rag,k,run):
    user=build_test_user(qid,rag,k)
    if run=="run2": user=user+"\n\n"+OUTPUT_CONSTRAINT
    messages=[{"role":"system","content":SYSTEM_PROMPT},{"role":"user","content":user}]
    prompt=tok.apply_chat_template(messages,tokenize=False,add_generation_prompt=True)
    inputs=tok(prompt,return_tensors="pt",truncation=True,max_length=6144,add_special_tokens=False)
    device=next(model.parameters()).device
    inputs={kk:vv.to(device) for kk,vv in inputs.items()}
    plen=inputs["input_ids"].shape[1]

    kwargs=dict(
        max_new_tokens=500 if run=="run1" else 350,
        do_sample=False,num_beams=1,
        pad_token_id=tok.eos_token_id,
        repetition_penalty=1.15 if run=="run1" else 1.20,
    )
    if run=="run2":
        kwargs["no_repeat_ngram_size"]=6
        stopids=list(dict.fromkeys([
            tok.eos_token_id,
            tok.convert_tokens_to_ids("<end_of_turn>")
        ]))
        kwargs["eos_token_id"]=stopids
        kwargs["stopping_criteria"]=StoppingCriteriaList([FirstValidJSONStoppingCriteria(tok,plen)])

    out=model.generate(**inputs,**kwargs)
    gen=out[0][plen:]
    raw=tok.decode(gen,skip_special_tokens=False)
    parsed,jtxt=find_first_complete_json(raw)
    maxn=kwargs["max_new_tokens"]
    hitmax=len(gen)>=maxn
    success=(parsed is not None and not hitmax)
    return {
        "case_id":qid,"run":run,"k":k,
        "raw_generation":raw,
        "first_valid_json_text":jtxt,
        "parsed_prediction":parsed,
        "technical_success":bool(success),
        "generated_tokens":int(len(gen)),
        "hit_max_tokens":bool(hitmax),
        "parse_status":"SUCCESS" if parsed is not None else "PARSE_FAILURE",
    }

def run_condition(name,model,tok,rag,k,run):
    path=os.path.join(OUTPUT_DIR,f"{name}.json")
    results={}
    if os.path.exists(path):
        with open(path,encoding="utf-8") as f:results=json.load(f)
        results={norm_id(a):b for a,b in results.items()}
        print(f"[RESUME] {name}: {len(results)}/101")
    start=time.time(); done0=len(results)
    for n,qid in enumerate(TEST_IDS,1):
        if qid in results:continue
        one=time.time()
        try:res=generate_case(model,tok,qid,rag,k,run)
        except Exception as e:
            res={"case_id":qid,"run":run,"k":k,"raw_generation":None,
                 "parsed_prediction":None,"technical_success":False,
                 "generated_tokens":0,"hit_max_tokens":False,
                 "parse_status":"GENERATION_EXCEPTION","exception":repr(e)}
        results[qid]=res
        with open(path,"w",encoding="utf-8") as f:json.dump(results,f,indent=2,ensure_ascii=False)

        completed=len(results)-done0
        elapsed=time.time()-start
        remaining=101-len(results)
        secper=elapsed/max(completed,1)
        eta=secper*remaining
        print(f"\r{name}: {len(results):3d}/101 | elapsed {fmt_time(elapsed)} | ETA {fmt_time(eta)} | last {time.time()-one:.1f}s",end="",flush=True)
    print(f"\n[TIME] {name}: {fmt_time(time.time()-start)}")
    return results


## 8. Frozen MedGemma — all 6 conditions


In [9]:
# ============================================================
# CORRECTED MEDGEMMA QLORA TOKEN-LENGTH AUDIT
# ============================================================

import json
import numpy as np
import pandas as pd

MAX_LENGTH = 6144

TRAIN_JSONL = "/kaggle/working/final_12_condition_llm_study/train_248_new_qlora.jsonl"
VAL_JSONL   = "/kaggle/working/final_12_condition_llm_study/val_28_new_qlora.jsonl"


# ------------------------------------------------------------
# 1. Load data
# ------------------------------------------------------------

def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [
            json.loads(line)
            for line in f
            if line.strip()
        ]


train_rows = load_jsonl(TRAIN_JSONL)
val_rows   = load_jsonl(VAL_JSONL)

print("Train examples:", len(train_rows))
print("Val examples:  ", len(val_rows))

assert len(train_rows) == 248
assert len(val_rows) == 28


# ------------------------------------------------------------
# 2. Safe chat-template -> text -> tokens
# ------------------------------------------------------------

def render_chat(messages, add_generation_prompt=False):

    rendered = tokenizer.apply_chat_template(
        messages,
        tokenize=False,          # IMPORTANT
        add_generation_prompt=add_generation_prompt
    )

    # Some multimodal/chat processors can return unusual structures.
    assert isinstance(rendered, str), (
        f"Expected rendered chat to be str, got {type(rendered)}"
    )

    return rendered


def tokenize_text(text):

    encoded = tokenizer(
        text,
        add_special_tokens=False,
        truncation=False,
        return_attention_mask=False
    )

    ids = encoded["input_ids"]

    # Handle accidental batch dimension
    if (
        len(ids) > 0
        and isinstance(ids[0], (list, tuple))
    ):
        assert len(ids) == 1
        ids = ids[0]

    return ids


# ------------------------------------------------------------
# 3. First inspect ONE example before running full audit
# ------------------------------------------------------------

example = train_rows[0]

print("\n" + "=" * 70)
print("RAW STRUCTURE CHECK")
print("=" * 70)

print("Top-level keys:")
print(example.keys())

print("\nNumber of messages:")
print(len(example["messages"]))

for i, msg in enumerate(example["messages"]):
    print(
        i,
        "role =", msg.get("role"),
        "content type =", type(msg.get("content")),
        "content chars =", len(str(msg.get("content")))
    )


full_text = render_chat(
    example["messages"],
    add_generation_prompt=False
)

prompt_text = render_chat(
    example["messages"][:-1],
    add_generation_prompt=True
)

full_ids = tokenize_text(full_text)
prompt_ids = tokenize_text(prompt_text)

print("\nRendered full-text characters:", len(full_text))
print("Rendered prompt characters:", len(prompt_text))

print("FULL tokens:", len(full_ids))
print("PROMPT tokens:", len(prompt_ids))

print("\nFirst 500 characters of rendered conversation:")
print(full_text[:500])


# Hard sanity checks
assert len(full_ids) > 100, (
    "Something is still wrong: complete conversation "
    f"has only {len(full_ids)} tokens."
)

assert len(prompt_ids) > 50, (
    "Something is still wrong: prompt "
    f"has only {len(prompt_ids)} tokens."
)

print("\n✓ Basic tokenization sanity check passed.")


# ------------------------------------------------------------
# 4. Audit each example
# ------------------------------------------------------------

def audit_example(row, split, index):

    messages = row["messages"]

    assert messages[-1]["role"] == "assistant"

    # Complete training conversation
    full_text = render_chat(
        messages,
        add_generation_prompt=False
    )

    full_ids = tokenize_text(full_text)

    # Input/prompt before assistant completion
    prompt_text = render_chat(
        messages[:-1],
        add_generation_prompt=True
    )

    prompt_ids = tokenize_text(prompt_text)

    # Raw target
    target_text = messages[-1]["content"]

    target_ids = tokenize_text(target_text)

    total_tokens = len(full_ids)
    prompt_tokens = len(prompt_ids)
    target_tokens = len(target_ids)

    # Note:
    # chat templates add control tokens, so
    # total != prompt + raw target exactly.
    truncated = total_tokens > MAX_LENGTH

    return {
        "split": split,
        "index": index,

        "case_id": row.get(
            "case_id",
            row.get("query_id", index)
        ),

        "total_tokens": total_tokens,
        "prompt_tokens": prompt_tokens,
        "target_tokens_raw": target_tokens,

        "tokens_over_limit":
            max(0, total_tokens - MAX_LENGTH),

        "full_sequence_fits_limit":
            total_tokens <= MAX_LENGTH,

        "prompt_alone_exceeds_limit":
            prompt_tokens >= MAX_LENGTH,

        "sequence_truncated_at_limit":
            truncated,
    }


audit_rows = []

for i, row in enumerate(train_rows):
    audit_rows.append(
        audit_example(row, "train", i)
    )

for i, row in enumerate(val_rows):
    audit_rows.append(
        audit_example(row, "val", i)
    )

audit_df = pd.DataFrame(audit_rows)


# ------------------------------------------------------------
# 5. Summary
# ------------------------------------------------------------

def summarize(df, name):

    print("\n" + "=" * 75)
    print(name)
    print("=" * 75)

    print("Examples:", len(df))

    print("\nTOTAL TOKENS")
    print(
        df["total_tokens"].describe(
            percentiles=[
                .50,
                .75,
                .90,
                .95,
                .99
            ]
        )
    )

    print("\nPROMPT TOKENS")
    print(
        df["prompt_tokens"].describe(
            percentiles=[
                .50,
                .75,
                .90,
                .95,
                .99
            ]
        )
    )

    print("\nTARGET TOKENS")
    print(
        df["target_tokens_raw"].describe(
            percentiles=[
                .50,
                .75,
                .90,
                .95,
                .99
            ]
        )
    )

    n = len(df)

    over = df["sequence_truncated_at_limit"].sum()
    prompt_over = df["prompt_alone_exceeds_limit"].sum()

    print(f"\n{MAX_LENGTH}-TOKEN LIMIT")

    print(
        f"Full sequences >1024: "
        f"{over}/{n} "
        f"({100 * over / n:.1f}%)"
    )

    print(
        f"Prompts alone >=1024: "
        f"{prompt_over}/{n} "
        f"({100 * prompt_over / n:.1f}%)"
    )


summarize(
    audit_df[audit_df["split"] == "train"],
    "TRAIN"
)

summarize(
    audit_df[audit_df["split"] == "val"],
    "VALIDATION"
)

summarize(
    audit_df,
    "COMBINED"
)


# ------------------------------------------------------------
# 6. Length thresholds
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("CONTEXT-LENGTH COVERAGE")
print("=" * 75)

for limit in [
    512,
    1024,
    1536,
    2048,
    3072,
    4096,
    6144,
    8192
]:

    fits = (
        audit_df["total_tokens"] <= limit
    ).mean() * 100

    print(
        f"{limit:>5} tokens : "
        f"{fits:6.2f}% sequences fit completely"
    )


# ------------------------------------------------------------
# 7. Longest cases
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("20 LONGEST EXAMPLES")
print("=" * 75)

display(
    audit_df.sort_values(
        "total_tokens",
        ascending=False
    )[
        [
            "split",
            "case_id",
            "total_tokens",
            "prompt_tokens",
            "target_tokens_raw",
            "tokens_over_limit"
        ]
    ].head(20)
)


# ------------------------------------------------------------
# 8. Cases where prompt alone consumes >=1024
# ------------------------------------------------------------

problem = audit_df[
    audit_df["prompt_alone_exceeds_limit"]
].copy()

print("\n" + "=" * 75)
print(f"PROMPTS >= {MAX_LENGTH} TOKENS")
print("=" * 75)

print(
    f"{len(problem)}/{len(audit_df)} examples"
)

if len(problem):
    display(
        problem.sort_values(
            "prompt_tokens",
            ascending=False
        )[
            [
                "split",
                "case_id",
                "total_tokens",
                "prompt_tokens",
                "target_tokens_raw"
            ]
        ].head(30)
    )


# ------------------------------------------------------------
# 9. Save
# ------------------------------------------------------------

audit_df.to_csv(
    "/kaggle/working/qlora_token_audit_CORRECTED.csv",
    index=False
)

print(
    "\nSaved:"
    "/kaggle/working/qlora_token_audit_CORRECTED.csv"
)

Train examples: 248
Val examples:   28

RAW STRUCTURE CHECK
Top-level keys:
dict_keys(['case_id', 'messages'])

Number of messages:
3
0 role = system content type = <class 'str'> content chars = 980
1 role = user content type = <class 'str'> content chars = 5705
2 role = assistant content type = <class 'str'> content chars = 1750

Rendered full-text characters: 8511
Rendered prompt characters: 6747
FULL tokens: 1858
PROMPT tokens: 1488

First 500 characters of rendered conversation:
<bos><start_of_turn>user
You are a dental clinical documentation assistant. You will be given a patient's Age, Sex, and Main appeal (chief complaint), along with similar PAST patient cases retrieved for reference.

CRITICAL: Retrieved cases are REFERENCE MATERIAL from OTHER patients. They are NOT facts about the current patient. Do not assume the current patient has the same findings, tooth numbers, diagnoses, or treatments unless the current patient's own information genuinely supports it.

Rul

✓ Basic to

,split,case_id,total_tokens,prompt_tokens,target_tokens_raw,tokens_over_limit
77,train,243,11466,10990,474,5322
163,train,635,8222,7758,462,2078
153,train,538,7678,7526,150,1534
118,train,490,7190,6283,905,1046
101,train,383,6850,4786,2062,706
16,train,253,6817,6705,110,673
220,train,294,6730,5702,1026,586
113,train,505,6690,3392,3296,546
114,train,436,6571,5540,1029,427
75,train,599,6241,6113,126,97



PROMPTS >= 6144 TOKENS
5/276 examples


,split,case_id,total_tokens,prompt_tokens,target_tokens_raw
77,train,243,11466,10990,474
163,train,635,8222,7758,462
153,train,538,7678,7526,150
16,train,253,6817,6705,110
118,train,490,7190,6283,905



Saved:/kaggle/working/qlora_token_audit_CORRECTED.csv


In [10]:
# ============================================================
# CHECK HOW MANY REFERENCE CASES SURVIVE THE CURRENT TOKEN WINDOW
# ============================================================

import re

def reference_survival(row, split, index, max_length=MAX_LENGTH):

    messages = row["messages"]

    # Chat-formatted full prompt WITHOUT assistant target
    prompt_ids = tokenizer.apply_chat_template(
        messages[:-1],
        tokenize=True,
        add_generation_prompt=True
    ).input_ids # Extract input_ids

    # Simulate right-side truncation
    kept_ids = prompt_ids[:max_length]

    kept_text = tokenizer.decode(
        kept_ids,
        skip_special_tokens=False
    )

    # Detect reference headings remaining after truncation
    refs_found = re.findall(
        r"REFERENCE CASE\s*(\d+)",
        kept_text,
        flags=re.IGNORECASE
    )

    return {
        "split": split,
        "index": index,
        "case_id": row.get("case_id", row.get("query_id", index)),
        "prompt_tokens": len(prompt_ids),
        "prompt_exceeds_limit": len(prompt_ids) > max_length,
        "reference_headings_surviving": len(set(refs_found)),
        "reference_numbers_surviving": ",".join(
            sorted(set(refs_found), key=int)
        )
    }


survival_rows = []

for i, row in enumerate(train_rows):
    survival_rows.append(
        reference_survival(row, "train", i)
    )

for i, row in enumerate(val_rows):
    survival_rows.append(
        reference_survival(row, "val", i)
    )

survival_df = pd.DataFrame(survival_rows)


print("=" * 70)
print("REFERENCE SURVIVAL AT MAX_LENGTH =", MAX_LENGTH)
print("=" * 70)

print(
    survival_df[
        "reference_headings_surviving"
    ].value_counts().sort_index()
)

print("\nPercent retaining all 5 reference headings:")

pct_all5 = (
    survival_df["reference_headings_surviving"].eq(5).mean() * 100
)

print(f"{pct_all5:.1f}%")


print("\nExamples losing one or more references:")

lost_refs = survival_df[
    survival_df["reference_headings_surviving"] < 5
].copy()

display(
    lost_refs.sort_values(
        "prompt_tokens",
        ascending=False
    ).head(30)
)


survival_df.to_csv(
    "/kaggle/working/qlora_reference_survival_audit.csv",
    index=False
)


REFERENCE SURVIVAL AT MAX_LENGTH = 6144
reference_headings_surviving
3      2
4      2
5    272
Name: count, dtype: int64

Percent retaining all 5 reference headings:
98.6%

Examples losing one or more references:


,split,index,case_id,prompt_tokens,prompt_exceeds_limit,reference_headings_surviving,reference_numbers_surviving
77,train,77,243,10990,True,3,"1,2,3"
163,train,163,635,7758,True,3,"1,2,3"
153,train,153,538,7526,True,4,"1,2,3,4"
16,train,16,253,6705,True,4,"1,2,3,4"


In [11]:
# Each condition is resume-safe. Estimated remaining time updates after every patient.
from pathlib import Path
from google.colab import files
import shutil

# ============================================================
# RESTORE THE TWO CONDITIONS YOU ALREADY HAVE
# ============================================================

OUTPUT_DIR = Path(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXISTING_FILES = {
    "FROZEN_NONE_RUN1": Path("/content/FROZEN_NONE_RUN1.json"),
    "FROZEN_BEFORE_K5_RUN1": Path("/content/FROZEN_BEFORE_K5_RUN1.json"),
}

for condition_name, uploaded_path in EXISTING_FILES.items():

    destination = OUTPUT_DIR / f"{condition_name}.json"

    if uploaded_path.exists():
        shutil.copy2(uploaded_path, destination)
        print(f"✓ Restored {condition_name}")
        print(f"  {uploaded_path} -> {destination}")
    else:
        print(f"✗ Missing uploaded file: {uploaded_path}")

✓ Restored FROZEN_NONE_RUN1
  /content/FROZEN_NONE_RUN1.json -> /kaggle/working/final_12_condition_llm_study/FROZEN_NONE_RUN1.json
✓ Restored FROZEN_BEFORE_K5_RUN1
  /content/FROZEN_BEFORE_K5_RUN1.json -> /kaggle/working/final_12_condition_llm_study/FROZEN_BEFORE_K5_RUN1.json


In [12]:
# ============================================================
# DOWNLOAD HELPER
# ============================================================

def download_condition(condition_name):
    path = OUTPUT_DIR / f"{condition_name}.json"

    if not path.exists():
        raise FileNotFoundError(f"Could not find: {path}")

    print(f"\nDownloading {path.name} ...")
    files.download(str(path))


# ============================================================
# LOAD/RESUME FIRST TWO
# These should detect the existing 101 predictions and
# therefore should NOT regenerate them.
# ============================================================

F_Z_R1 = run_condition(
    "FROZEN_NONE_RUN1",
    frozen, tokenizer, None, 0, "run1"
)
download_condition("FROZEN_NONE_RUN1")


F_B_R1 = run_condition(
    "FROZEN_BEFORE_K5_RUN1",
    frozen, tokenizer, before, 5, "run1"
)
download_condition("FROZEN_BEFORE_K5_RUN1")


# ============================================================
# GENERATE THE FOUR MISSING CONDITIONS
# Each downloads immediately after completion
# ============================================================

F_A_R1 = run_condition(
    "FROZEN_AFTER_K5_RUN1",
    frozen, tokenizer, after, 5, "run1"
)
download_condition("FROZEN_AFTER_K5_RUN1")


F_Z_R2 = run_condition(
    "FROZEN_NONE_RUN2",
    frozen, tokenizer, None, 0, "run2"
)
download_condition("FROZEN_NONE_RUN2")


F_B_R2 = run_condition(
    "FROZEN_BEFORE_K5_RUN2",
    frozen, tokenizer, before, 5, "run2"
)
download_condition("FROZEN_BEFORE_K5_RUN2")


F_A_R2 = run_condition(
    "FROZEN_AFTER_K5_RUN2",
    frozen, tokenizer, after, 5, "run2"
)
download_condition("FROZEN_AFTER_K5_RUN2")

[RESUME] FROZEN_NONE_RUN1: 101/101

[TIME] FROZEN_NONE_RUN1: 0s



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

[RESUME] FROZEN_BEFORE_K5_RUN1: 101/101

[TIME] FROZEN_BEFORE_K5_RUN1: 0s



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

... (8 outputs left out)

<IPython.core.display.Javascript object>

FROZEN_AFTER_K5_RUN2: 101/101 | elapsed 47m 15s | ETA 0s | last 28.3s
[TIME] FROZEN_AFTER_K5_RUN2: 47m 15s



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 9. Free frozen model before QLoRA training


In [13]:
with timed("Free frozen model"):
    del frozen
    gc.collect()
    if torch.cuda.is_available():torch.cuda.empty_cache()



[TIMER START] Free frozen model
[TIME] Free frozen model: 0.4s (0.01 min)


## 10. New QLoRA training — same audited hyperparameters + live ETA


In [16]:
from peft import prepare_model_for_kbit_training

class TrainingETACallback(TrainerCallback):
    def __init__(self): self.start=None
    def on_train_begin(self,args,state,control,**kwargs):
        self.start=time.time()
        print(f"QLoRA started | max steps {state.max_steps} | max epochs {args.num_train_epochs}")
    def on_step_end(self,args,state,control,**kwargs):
        if not self.start or state.global_step<1:return
        e=time.time()-self.start
        eta=e/state.global_step*max(state.max_steps-state.global_step,0)
        print(f"\rStep {state.global_step}/{state.max_steps} | epoch {state.epoch:.2f} | elapsed {fmt_time(e)} | ETA {fmt_time(eta)}",end="",flush=True)
    def on_evaluate(self,args,state,control,metrics=None,**kwargs):
        print(f"\nValidation epoch {state.epoch:.2f} | eval_loss={metrics.get('eval_loss',float('nan')):.6f} | elapsed {fmt_time(time.time()-self.start)}")
    def on_train_end(self,args,state,control,**kwargs):
        print(f"\n[TIME] QLoRA training: {fmt_time(time.time()-self.start)}")

with timed("Load QLoRA base model + prepare SFT"):
    qtok=AutoTokenizer.from_pretrained(BASE_MODEL)
    if qtok.pad_token is None: qtok.pad_token=qtok.eos_token

    # Match the fixed QLoRA notebook: T4-safe float16 compute.
    q_bnb=BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    qmodel=AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,device_map="auto",quantization_config=q_bnb,
        attn_implementation="sdpa",
    )

    # IMPORTANT FIX from the audited notebook.
    qmodel=prepare_model_for_kbit_training(qmodel)
    qmodel.gradient_checkpointing_enable(
        gradient_checkpointing_kwargs={"use_reentrant":False}
    )
    qmodel.config.use_cache=False

    lora=LoraConfig(
        r=16,lora_alpha=64,lora_dropout=0.08,
        target_modules=["q_proj","k_proj","v_proj","o_proj"],
        bias="none",task_type="CAUSAL_LM"
    )

    tr=load_dataset("json",data_files=TRAIN_JSONL)["train"]
    va=load_dataset("json",data_files=VAL_JSONL)["train"]

    def to_pc(ex):
        m=ex["messages"]
        prompt=qtok.apply_chat_template(m[:-1],tokenize=False,add_generation_prompt=True)
        full=qtok.apply_chat_template(m,tokenize=False,add_generation_prompt=False)
        completion=full[len(prompt):] if full.startswith(prompt) else m[-1]["content"]+qtok.eos_token
        return {"prompt":prompt,"completion":completion}

    tr=tr.map(to_pc,remove_columns=tr.column_names)
    va=va.map(to_pc,remove_columns=va.column_names)
    print("Train:",len(tr),"Val:",len(va))

# Same text-only Gemma3 fix as the audited notebook, implemented in the collator
# so every padded batch receives token_type_ids=0.
class Gemma3TextDataCollator(DataCollatorForLanguageModeling):
    def __call__(self,examples):
        batch=super().__call__(examples)
        batch["token_type_ids"]=torch.zeros_like(batch["input_ids"])
        return batch

collator=Gemma3TextDataCollator(
    pad_token_id=qtok.pad_token_id,
    #completion_only_loss=True, # This argument belongs to SFTConfig, not DataCollatorForLanguageModeling
)

QDIR=os.path.join(OUTPUT_DIR,"NEW_QLORA_R16_A64")
cfg=SFTConfig(
    output_dir=QDIR,
    max_length=6144,                       # match audited notebook
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,        # match audited notebook
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant":False},
    learning_rate=5e-5,                    # match audited notebook
    num_train_epochs=10,
    lr_scheduler_type="cosine",
    #warmup_ratio=0.1,                      # match audited notebook
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    # IMPORTANT FIX from audited notebook: avoid AMP GradScaler/BFloat16 failure.
    fp16=False,
    bf16=False,
    optim="paged_adamw_8bit",
    report_to="none",
    packing=False,
    seed=42,
    completion_only_loss=True,
    loss_type="nll",
)

trainer=SFTTrainer(
    model=qmodel,args=cfg,
    train_dataset=tr,eval_dataset=va,
    data_collator=collator,
    processing_class=qtok,
    peft_config=lora,
    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=3,early_stopping_threshold=0.0),
        TrainingETACallback()
    ],
)

# Fail-fast sanity checks BEFORE a long training run.
preview=next(iter(trainer.get_train_dataloader()))
assert "token_type_ids" in preview
assert torch.all(preview["token_type_ids"]==0)
assert int((preview["labels"]!=-100).sum())>0, "No completion tokens contribute to loss"
assert int((preview["labels"]==-100).sum())>0, "Prompt is not masked: completion-only loss is broken"
print("[PASS] token_type_ids=0")
print("[PASS] completion-only labels contain both masked prompt and supervised completion tokens")

train_wall=time.time()
trainer.train()
print("[TIME] trainer.train():",fmt_time(time.time()-train_wall))

trainer.save_model(QDIR)
qtok.save_pretrained(QDIR)
qmodel=trainer.model
qmodel.eval()
qmodel.config.use_cache=True
print("[PASS] use_cache re-enabled for inference")
print("Best checkpoint:",trainer.state.best_model_checkpoint)
print("Best validation loss:",trainer.state.best_metric)



[TIMER START] Load QLoRA base model + prepare SFT


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

Train: 248 Val: 28
[TIME] Load QLoRA base model + prepare SFT: 20.8s (0.35 min)


Adding EOS to train dataset:   0%|          | 0/248 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/248 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/248 [00:00<?, ? examples/s]

... (6 outputs left out)

Dropping fully masked examples from eval dataset:   0%|          | 0/28 [00:00<?, ? examples/s]

[PASS] token_type_ids=0
[PASS] completion-only labels contain both masked prompt and supervised completion tokens
QLoRA started | max steps 160 | max epochs 10
Step 1/160 | epoch 0.07 | elapsed 25s | ETA 1h 7m 7s

Epoch,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
1,1.484484,1.429571,1.461895,0.703069,824938.000000
2,1.389196,1.361968,1.355649,0.712888,1649876.000000
3,1.381014,1.323037,1.307267,0.716699,2474814.000000
4,1.247862,1.291538,1.243000,0.720954,3299752.000000
5,1.236396,1.270949,1.196049,0.723747,4124690.000000
6,1.164653,1.258179,1.155804,0.725809,4949628.000000
7,1.106165,1.250785,1.160845,0.727160,5774566.000000
8,1.203901,1.248995,1.141228,0.727779,6599504.000000
9,1.133217,1.249000,1.138850,0.727148,7424442.000000
10,1.023044,1.248970,1.138918,0.727458,8249380.000000


Step 16/160 | epoch 1.00 | elapsed 8m 58s | ETA 1h 20m 45s
Validation epoch 1.00 | eval_loss=1.429571 | elapsed 9m 17s
Step 32/160 | epoch 2.00 | elapsed 18m 16s | ETA 1h 13m 6s
Validation epoch 2.00 | eval_loss=1.361968 | elapsed 18m 36s
Step 48/160 | epoch 3.00 | elapsed 27m 34s | ETA 1h 4m 20s
Validation epoch 3.00 | eval_loss=1.323037 | elapsed 27m 53s
Step 64/160 | epoch 4.00 | elapsed 36m 52s | ETA 55m 18s
Validation epoch 4.00 | eval_loss=1.291538 | elapsed 37m 11s
Step 80/160 | epoch 5.00 | elapsed 46m 10s | ETA 46m 10s
Validation epoch 5.00 | eval_loss=1.270949 | elapsed 46m 29s
Step 96/160 | epoch 6.00 | elapsed 55m 28s | ETA 36m 58s
Validation epoch 6.00 | eval_loss=1.258179 | elapsed 55m 47s
Step 112/160 | epoch 7.00 | elapsed 1h 4m 46s | ETA 27m 45s
Validation epoch 7.00 | eval_loss=1.250785 | elapsed 1h 5m 5s
Step 128/160 | epoch 8.00 | elapsed 1h 14m 4s | ETA 18m 31s
Validation epoch 8.00 | eval_loss=1.248995 | elapsed 1h 14m 24s
Step 144/160 | epoch 9.00 | elapsed 1h 23

In [22]:
# ============================================================
# QLoRA FIX AUDIT — MUST ALL PASS BEFORE INFERENCE
# ============================================================
print("="*72)
print("QLORA FIX AUDIT")
print("="*72)
assert cfg.max_length == 6144
assert cfg.gradient_accumulation_steps == 16
assert abs(cfg.learning_rate - 5e-5) < 1e-12
assert cfg.gradient_checkpointing_kwargs.get("use_reentrant") is False
assert cfg.fp16 is False and cfg.bf16 is False
assert cfg.completion_only_loss is True
assert qmodel.config.use_cache is True
assert lora.r == 16 and lora.lora_alpha == 64
assert abs(lora.lora_dropout - 0.08) < 1e-12
assert set(lora.target_modules)=={"q_proj","k_proj","v_proj","o_proj"}
assert len(train_ids)==248 and len(val_ids)==28
assert set(train_ids).isdisjoint(set(TEST_IDS))
assert set(val_ids).isdisjoint(set(TEST_IDS))
print("[PASS] prepare_model_for_kbit_training applied before PEFT/SFT")
print("[PASS] NF4 + double quant + float16 compute")
print("[PASS] non-reentrant gradient checkpointing")
print("[PASS] token_type_ids=0 text-only batches")
print("[PASS] fp16=False, bf16=False trainer flags")
print("[PASS] completion-only loss")
print("[PASS] r16 / alpha64 / dropout0.08 / q-k-v-o")
print("[PASS] 248/28 split and zero test-query overlap")
print("[PASS] use_cache=True for inference")
print("="*72)


QLORA FIX AUDIT
[PASS] prepare_model_for_kbit_training applied before PEFT/SFT
[PASS] NF4 + double quant + float16 compute
[PASS] non-reentrant gradient checkpointing
[PASS] token_type_ids=0 text-only batches
[PASS] fp16=False, bf16=False trainer flags
[PASS] completion-only loss
[PASS] r16 / alpha64 / dropout0.08 / q-k-v-o
[PASS] 248/28 split and zero test-query overlap
[PASS] use_cache=True for inference


## 11. QLoRA — all 6 conditions using the same newly trained adapter


In [23]:
Q_Z_R1 = run_condition("QLORA_NONE_RUN1", qmodel,qtok,None,0,"run1")
Q_B_R1 = run_condition("QLORA_BEFORE_K5_RUN1",qmodel,qtok,before,5,"run1")
Q_A_R1 = run_condition("QLORA_AFTER_K5_RUN1", qmodel,qtok,after,5,"run1")

Q_Z_R2 = run_condition("QLORA_NONE_RUN2", qmodel,qtok,None,0,"run2")
Q_B_R2 = run_condition("QLORA_BEFORE_K5_RUN2",qmodel,qtok,before,5,"run2")
Q_A_R2 = run_condition("QLORA_AFTER_K5_RUN2", qmodel,qtok,after,5,"run2")


[RESUME] QLORA_NONE_RUN1: 101/101

[TIME] QLORA_NONE_RUN1: 0s
[RESUME] QLORA_BEFORE_K5_RUN1: 101/101

[TIME] QLORA_BEFORE_K5_RUN1: 0s
[RESUME] QLORA_AFTER_K5_RUN1: 101/101

[TIME] QLORA_AFTER_K5_RUN1: 0s
[RESUME] QLORA_NONE_RUN2: 101/101

[TIME] QLORA_NONE_RUN2: 0s
[RESUME] QLORA_BEFORE_K5_RUN2: 101/101

[TIME] QLORA_BEFORE_K5_RUN2: 0s
[RESUME] QLORA_AFTER_K5_RUN2: 101/101

[TIME] QLORA_AFTER_K5_RUN2: 0s


## 12. One corrected evaluator for all 12 conditions


In [24]:
with timed("Load evaluation packages"):
    import evaluate, nltk
    # Allow proxied fetches for NLTK data downloads in Colab environments
    nltk.pathsec.ALLOW_PROXIED_FETCH = True
    nltk.download("wordnet",quiet=True)
    nltk.download("omw-1.4",quiet=True)
    nltk.download("punkt",quiet=True) # Required for NLTK tokenizers used by METEOR
    bleu_m=evaluate.load("bleu");rouge_m=evaluate.load("rouge");meteor_m=evaluate.load("meteor")

def normalize(text):
    if text is None:return ""
    try:
        if pd.isna(text):return ""
    except:pass
    text=str(text).replace("&#39;","'").replace("&amp;","&").replace("&quot;",'"')
    return re.sub(r"\s+"," ",text).strip().lower()

TOOTH_WORD_RE=re.compile(r"\btooth\s*[*#:]?\s*(1[1-8]|2[1-8]|3[1-8]|4[1-8])\b",re.I)
TOOTH_STAR_RE=re.compile(r"(?<!\d)\*\s*(1[1-8]|2[1-8]|3[1-8]|4[1-8])\b",re.I)
ICD10_RE=re.compile(r"\b([A-Za-z])[\s-]?(\d{2}(?:\.\d{1,4})?)\b",re.I)

def fdi(t):
    t=normalize(t);return set(TOOTH_WORD_RE.findall(t))|set(TOOTH_STAR_RE.findall(t))
def icd(t):
    return {a.upper()+b for a,b in ICD10_RE.findall(normalize(t))}
def set_f1(a,b):
    if not a and not b:return None
    if not a or not b:return 0.
    tp=len(a&b);p=tp/len(b);r=tp/len(a)
    return 2*p*r/(p+r) if p+r else 0.
def gt(q):
    r=full[q];return {f:clean(r.get(f),"") for f in TARGET_FIELDS}
def report(x):return " ".join(clean(x.get(f),"") for f in TARGET_FIELDS)
def grams4(t):
    w=re.findall(r"\w+",normalize(t));return set(tuple(w[i:i+4]) for i in range(max(0,len(w)-3)))
def evidence(rag,q,k):
    if rag is None or k==0:return ""
    parts=[]
    for ex in rag[q]["retrieved_exemplars"][:k]:
        rr=exemplar_record(ex)
        parts.append(" ".join(clean(rr.get(f),"") for f in REFERENCE_FIELDS))
    return " ".join(parts)

def evaluate_condition(outputs,rag,k):
    success=[q for q in TEST_IDS if outputs[q].get("technical_success") and isinstance(outputs[q].get("parsed_prediction"),dict)]
    refs=[];hyps=[];fvals=[];ivals=[]
    TP=FP=FN=0; hall=[];hn=hd=0;copy=[];fab=[]
    for q in success:
        p=outputs[q]["parsed_prediction"];g=gt(q)
        pt,gtxt=report(p),report(g);refs.append(gtxt);hyps.append(pt)
        pf,gf=fdi(pt),fdi(gtxt)
        # ICD is diagnosis-field only.
        pi,gi=icd(p.get("Diagnosis","")),icd(g.get("Diagnosis",""))
        x=set_f1(gf,pf);y=set_f1(gi,pi)
        if x is not None:fvals.append(x)
        if y is not None:ivals.append(y)
        TP+=len(pf&gf);FP+=len(pf-gf);FN+=len(gf-pf)

        pe={("FDI",x) for x in pf}|{("ICD",x) for x in pi}
        ge={("FDI",x) for x in gf}|{("ICD",x) for x in gi}
        if pe:
            u=len(pe-ge);hall.append(u/len(pe));hn+=u;hd+=len(pe)

        ev=evidence(rag,q,k)
        if k>0:
            p4,e4=grams4(pt),grams4(ev)
            if p4:copy.append(len((p4&e4)-grams4(gtxt))/len(p4))
            et=fdi(ev)
            if pf:fab.append(len(pf-gf-et)/len(pf))
        else:
            # No retrieval: fabricated tooth means unsupported by GT.
            if pf:fab.append(len(pf-gf)/len(pf))

    if hyps:
        B=bleu_m.compute(predictions=hyps,references=[[r] for r in refs])["bleu"]
        R=rouge_m.compute(predictions=hyps,references=refs)["rougeL"]
        M=meteor_m.compute(predictions=hyps,references=refs)["meteor"]
    else:B=R=M=np.nan
    pr=TP/(TP+FP) if TP+FP else 0;rc=TP/(TP+FN) if TP+FN else 0
    micro=2*pr*rc/(pr+rc) if pr+rc else 0
    return {
        "N_total":101,"N_success":len(success),"Technical_success":len(success)/101,
        "BLEU4":B,"ROUGE_L":R,"METEOR":M,
        "FDI_macro_F1":np.mean(fvals) if fvals else np.nan,"FDI_n":len(fvals),
        "FDI_micro_F1":micro,
        "ICD_macro_F1":np.mean(ivals) if ivals else np.nan,"ICD_n":len(ivals),
        "Unsupported_entity_patient_mean":np.mean(hall) if hall else np.nan,
        "Unsupported_entity_micro":hn/hd if hd else np.nan,
        "Retrieval_copy_rate":np.mean(copy) if copy else np.nan, # N/A for no-RAG
        "Fabricated_tooth_rate":np.mean(fab) if fab else np.nan,
        "Max_token_failures":sum(bool(outputs[q].get("hit_max_tokens")) for q in TEST_IDS),
    }


[TIMER START] Load evaluation packages
[TIME] Load evaluation packages: 4.2s (0.07 min)


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


## 13. Final 12-condition table


In [25]:
with timed("Evaluate all 12 conditions"):
    CONDITIONS={
      "Frozen | None | Run1":(F_Z_R1,None,0),
      "Frozen | Before | Run1":(F_B_R1,before,5),
      "Frozen | After | Run1":(F_A_R1,after,5),
      "Frozen | None | Run2":(F_Z_R2,None,0),
      "Frozen | Before | Run2":(F_B_R2,before,5),
      "Frozen | After | Run2":(F_A_R2,after,5),
      "QLoRA | None | Run1":(Q_Z_R1,None,0),
      "QLoRA | Before | Run1":(Q_B_R1,before,5),
      "QLoRA | After | Run1":(Q_A_R1,after,5),
      "QLoRA | None | Run2":(Q_Z_R2,None,0),
      "QLoRA | Before | Run2":(Q_B_R2,before,5),
      "QLoRA | After | Run2":(Q_A_R2,after,5),
    }
    RES={n:evaluate_condition(*v) for n,v in CONDITIONS.items()}
    final=pd.DataFrame(RES).T
    display(final)
    final.to_csv(os.path.join(OUTPUT_DIR,"FINAL_12_CONDITION_RESULTS.csv"))
    with open(os.path.join(OUTPUT_DIR,"FINAL_12_CONDITION_RESULTS.json"),"w") as f:
        json.dump(RES,f,indent=2)



[TIMER START] Evaluate all 12 conditions


,N_total,N_success,Technical_success,BLEU4,ROUGE_L,METEOR,FDI_macro_F1,FDI_n,FDI_micro_F1,ICD_macro_F1,ICD_n,Unsupported_entity_patient_mean,Unsupported_entity_micro,Retrieval_copy_rate,Fabricated_tooth_rate,Max_token_failures
Frozen | None | Run1,101.0,0.0,0.000000,NaN,NaN,NaN,NaN,0.0,0.000000,NaN,0.0,NaN,NaN,NaN,NaN,99.0
Frozen | Before | Run1,101.0,0.0,0.000000,NaN,NaN,NaN,NaN,0.0,0.000000,NaN,0.0,NaN,NaN,NaN,NaN,77.0
Frozen | After | Run1,101.0,0.0,0.000000,NaN,NaN,NaN,NaN,0.0,0.000000,NaN,0.0,NaN,NaN,NaN,NaN,76.0
Frozen | None | Run2,101.0,0.0,0.000000,NaN,NaN,NaN,NaN,0.0,0.000000,NaN,0.0,NaN,NaN,NaN,NaN,101.0
Frozen | Before | Run2,101.0,0.0,0.000000,NaN,NaN,NaN,NaN,0.0,0.000000,NaN,0.0,NaN,NaN,NaN,NaN,87.0
Frozen | After | Run2,101.0,0.0,0.000000,NaN,NaN,NaN,NaN,0.0,0.000000,NaN,0.0,NaN,NaN,NaN,NaN,98.0
QLoRA | None | Run1,101.0,100.0,0.990099,1.246014e-07,0.061763,0.042762,0.000000,95.0,0.000000,0.0,17.0,1.000000,1.000000,NaN,1.000000,1.0
QLoRA | Before | Run1,101.0,72.0,0.712871,1.226379e-02,0.174249,0.155473,0.072610,71.0,0.106383,0.0,25.0,0.840986,0.788732,0.447199,0.116279,29.0
QLoRA | After | Run1,101.0,78.0,0.772277,8.668128e-03,0.173677,0.154546,0.082538,76.0,0.078431,0.0,19.0,0.797386,0.815385,0.481085,0.039216,21.0
QLoRA | None | Run2,101.0,86.0,0.851485,2.969594e-05,0.079930,0.062937,0.000000,81.0,0.000000,0.0,13.0,1.000000,1.000000,NaN,NaN,8.0


[TIME] Evaluate all 12 conditions: 20.1s (0.34 min)


## 14. Controlled deltas: RAG, reranking, QLoRA and Run-2


In [26]:
metrics=[
 "Technical_success","ROUGE_L","METEOR","FDI_macro_F1","FDI_micro_F1",
 "ICD_macro_F1","Unsupported_entity_patient_mean","Unsupported_entity_micro",
 "Retrieval_copy_rate","Fabricated_tooth_rate"
]
pairs=[]

# Effect of reranking: After - Before
for gen in ["Frozen","QLoRA"]:
    for run in ["Run1","Run2"]:
        pairs.append((f"RERANK | {gen} | {run}",f"{gen} | Before | {run}",f"{gen} | After | {run}"))

# Effect of adding RAG before reranking: Before - None
for gen in ["Frozen","QLoRA"]:
    for run in ["Run1","Run2"]:
        pairs.append((f"RAG | {gen} | {run}",f"{gen} | None | {run}",f"{gen} | Before | {run}"))

# Effect of QLoRA: QLoRA - Frozen
for ctx in ["None","Before","After"]:
    for run in ["Run1","Run2"]:
        pairs.append((f"QLORA | {ctx} | {run}",f"Frozen | {ctx} | {run}",f"QLoRA | {ctx} | {run}"))

# Effect of robust inference: Run2 - Run1
for gen in ["Frozen","QLoRA"]:
    for ctx in ["None","Before","After"]:
        pairs.append((f"RUN2 | {gen} | {ctx}",f"{gen} | {ctx} | Run1",f"{gen} | {ctx} | Run2"))

rows=[]
for label,a,b in pairs:
    r={"Comparison":label,"A":a,"B":b}
    for m in metrics:r[m]=RES[b][m]-RES[a][m]
    rows.append(r)
delta=pd.DataFrame(rows)
display(delta)
delta.to_csv(os.path.join(OUTPUT_DIR,"CONTROLLED_DELTAS.csv"),index=False)


,Comparison,A,B,Technical_success,ROUGE_L,METEOR,FDI_macro_F1,FDI_micro_F1,ICD_macro_F1,Unsupported_entity_patient_mean,Unsupported_entity_micro,Retrieval_copy_rate,Fabricated_tooth_rate
0,RERANK | Frozen | Run1,Frozen | Before | Run1,Frozen | After | Run1,0.000000,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,NaN
1,RERANK | Frozen | Run2,Frozen | Before | Run2,Frozen | After | Run2,0.000000,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,NaN
2,RERANK | QLoRA | Run1,QLoRA | Before | Run1,QLoRA | After | Run1,0.059406,-0.000572,-0.000927,0.009928,-0.027952,0.0,-0.043601,0.026652,0.033886,-0.077063
3,RERANK | QLoRA | Run2,QLoRA | Before | Run2,QLoRA | After | Run2,0.079208,0.004261,-0.006392,0.009759,0.005403,0.0,-0.028214,-0.043548,0.019174,-0.099490
4,RAG | Frozen | Run1,Frozen | None | Run1,Frozen | Before | Run1,0.000000,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,NaN
5,RAG | Frozen | Run2,Frozen | None | Run2,Frozen | Before | Run2,0.000000,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,NaN
6,RAG | QLoRA | Run1,QLoRA | None | Run1,QLoRA | Before | Run1,-0.277228,0.112486,0.112711,0.072610,0.106383,0.0,-0.159014,-0.211268,NaN,-0.883721
7,RAG | QLoRA | Run2,QLoRA | None | Run2,QLoRA | Before | Run2,-0.089109,0.060178,0.065693,0.012051,0.022472,0.0,-0.032500,-0.034884,NaN,NaN
8,QLORA | None | Run1,Frozen | None | Run1,QLoRA | None | Run1,0.990099,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,NaN
9,QLORA | None | Run2,Frozen | None | Run2,QLoRA | None | Run2,0.851485,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,NaN


## 15. Common-patient Run-1 vs Run-2 comparisons


In [27]:
def paired_run1_run2(name,r1,r2):
    ids=[q for q in TEST_IDS if r1[q].get("technical_success") and r2[q].get("technical_success")]
    rows=[]
    for q in ids:
        row={"case_id":q}
        g=gt(q)
        for f in TARGET_FIELDS:
            row[f"GT_{f}"]=g[f]
            row[f"RUN1_{f}"]=r1[q]["parsed_prediction"][f]
            row[f"RUN2_{f}"]=r2[q]["parsed_prediction"][f]
        rows.append(row)
    df=pd.DataFrame(rows)
    path=os.path.join(OUTPUT_DIR,f"COMMON_PATIENT_{name}.csv")
    df.to_csv(path,index=False)
    print(name,":",len(df),"common technically successful patients ->",path)
    return df

paired_run1_run2("FROZEN_NONE",F_Z_R1,F_Z_R2)
paired_run1_run2("FROZEN_BEFORE",F_B_R1,F_B_R2)
paired_run1_run2("FROZEN_AFTER",F_A_R1,F_A_R2)
paired_run1_run2("QLORA_NONE",Q_Z_R1,Q_Z_R2)
paired_run1_run2("QLORA_BEFORE",Q_B_R1,Q_B_R2)
paired_run1_run2("QLORA_AFTER",Q_A_R1,Q_A_R2)


FROZEN_NONE : 0 common technically successful patients -> /kaggle/working/final_12_condition_llm_study/COMMON_PATIENT_FROZEN_NONE.csv
FROZEN_BEFORE : 0 common technically successful patients -> /kaggle/working/final_12_condition_llm_study/COMMON_PATIENT_FROZEN_BEFORE.csv
FROZEN_AFTER : 0 common technically successful patients -> /kaggle/working/final_12_condition_llm_study/COMMON_PATIENT_FROZEN_AFTER.csv
QLORA_NONE : 85 common technically successful patients -> /kaggle/working/final_12_condition_llm_study/COMMON_PATIENT_QLORA_NONE.csv
QLORA_BEFORE : 62 common technically successful patients -> /kaggle/working/final_12_condition_llm_study/COMMON_PATIENT_QLORA_BEFORE.csv
QLORA_AFTER : 70 common technically successful patients -> /kaggle/working/final_12_condition_llm_study/COMMON_PATIENT_QLORA_AFTER.csv


,case_id,GT_Oral Check,RUN1_Oral Check,RUN2_Oral Check,GT_Diagnosis,RUN1_Diagnosis,RUN2_Diagnosis,GT_Treatment plan,RUN1_Treatment plan,RUN2_Treatment plan,GT_Handle,RUN1_Handle,RUN2_Handle,GT_Doctor advices,RUN1_Doctor advices,RUN2_Doctor advices
0,378,tooth 16 (upper right first molar) temporarily...,tooth 26 (upper left first molar): crack,*26 the fillings are intact without loosenness...,tooth 16 (upper right first molar) tooth defec...,malformed teeth,*26 deep caries,tooth 16 (upper right first molar). filling tr...,not determined,*26 plan 1: filling repair,"6(1) Explain the condition, treatment plan, co...",the patient chose invisalign and decided to pa...,"*26 explain the condition, keep the whole body...",Do not bite hard objects and return to the cli...,Follow-up for discomfort.,Follow-up for discomfort.
1,219,tooth 13 (upper right canine) tooth 24 (upper ...,"the sutures did not fall off, the wound healed...",*The sutures did not fall off and the wound he...,tooth 13 (upper right canine) tooth 24 (upper ...,missing teeth,*Missing teeth*,tooth 13 (upper right canine) tooth 24 (upper ...,not determined,"*After extraction, the lower front tooth is re...","13 24 44 45 Gingival retraction, decay prepara...",routine disinfection and draping.,*Routine disinfection and draping. Rinse with ...,Follow-up for discomfort. Routine medical advi...,Follow-up for discomfort.,*Follow up for discomfort.
2,98,67 a loose white filling was seen in the inter...,* tooth 38 (lower left third molar): the too...,*There is an obvious dark brown stain on the p...,"tooth 36 (lower left first molar), tooth 37 (l...",chronic apical periodontitis,tooth 11 (upper central incisor): pigmented en...,"tooth 36 (lower left first molar), tooth 37 (l...",root canal treatment,resin filling,"36, 37 Clean the caries, remove part of the or...","Explain the condition, treatment plan, cost, p...","Explain the condition, treatment options, cost...",,Follow-up for discomfort.,Follow-up for discomfort.
3,414,tooth 14 (upper right first premolar) tooth 15...,*21 There is no obvious redness or swelling in...,"*There is an open socket in the mandible, whic...",tooth 32 (lower left lateral incisor) tooth 33...,Missing teeth,Mandibular third molars impacted teeth,tooth 32 (lower left lateral incisor) tooth 33...,"Full mouth inspection, cleaning, sealing, and ...",Extraction,"Explain the condition, treatment plan, cost, p...",Routine oral exam. Unscrew the healing cap and...,"Explain the condition, treatment options, pros...",Follow-up for discomfort.,"Avoid biting hard food, pay attention to oral ...","Avoid biting hard food, payattention to oral h..."
4,270,tooth 13 (upper right canine) the mesioproxima...,tooth 37 (lower left second molar) the buccal ...,*The buccal side of the upper right posterior ...,3 chronic apical periodontitis,tooth 37 (lower left second molar) dentinal hy...,Acute apical periododontitis,3 post crown restoration after root canal trea...,not determined,Pulmonary Pad Filling,"3. Explain the condition, keep the patient and...","Explain the condition, treatment options, cost...","Explain the condition, treatment options, cost...",Avoid biting hard objects. If pulp symptoms oc...,Follow-up for discomfort.,Follow-up for discomfort.
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
65,350,there is no obvious redness or swelling in the...,"the gums are red and swollen, and they bleed e...",*There is an open defect in the middle of the ...,dental caries missing teeth residual root (k 0...,periodontal disease,Malformation Malformation,implantation second stage restoration resin fi...,not determined,"Explain the condition, treatment options, cost...",Unscrew the healing cap and rinse and dry the ...,"Explain the condition, treatment plan, cost, p...",Local disinfection with iodophor and scraping....,Pay attention to oral hygiene and return to th...,Routine medical advice and follow-up consultat...,Pay attention to oral hygiene.
66,376,to

In [28]:
# ============================================================
# UNIVERSAL RUN-1 / RUN-2 SAFETY AUDIT
# These fixes apply to Frozen AND QLoRA, None/Before/After.
# ============================================================
print("="*72)
print("UNIVERSAL INFERENCE/EVALUATION AUDIT")
print("="*72)

all_outputs=[
    F_Z_R1,F_B_R1,F_A_R1,F_Z_R2,F_B_R2,F_A_R2,
    Q_Z_R1,Q_B_R1,Q_A_R1,Q_Z_R2,Q_B_R2,Q_A_R2
]
assert len(all_outputs)==12
for out in all_outputs:
    assert set(out)==set(TEST_IDS)
    for q,r in out.items():
        # A failed generation must NOT masquerade as a clinical prediction.
        if not r.get("technical_success"):
            assert not isinstance(r.get("parsed_prediction"),dict) or r.get("hit_max_tokens"), \
                f"{q}: failed case unexpectedly represented as ordinary clinical output"
        assert "raw_generation" in r
        assert "generated_tokens" in r
        assert "hit_max_tokens" in r
        assert "parse_status" in r

# Confirm empty-empty entity sets are excluded rather than rewarded.
assert set_f1(set(),set()) is None

# No-RAG copy metric must be undefined, not zero.
for n in ["Frozen | None | Run1","Frozen | None | Run2","QLoRA | None | Run1","QLoRA | None | Run2"]:
    assert pd.isna(RES[n]["Retrieval_copy_rate"])

print("[PASS] no top-1 fallback in any of 12 conditions")
print("[PASS] technical failures remain technical failures")
print("[PASS] raw generations retained")
print("[PASS] token-ceiling status retained")
print("[PASS] identical corrected evaluator across Run-1 and Run-2")
print("[PASS] empty-empty FDI/ICD does not receive F1=1")
print("[PASS] no-RAG retrieval-copy rate is N/A")
print("="*72)


UNIVERSAL INFERENCE/EVALUATION AUDIT
[PASS] no top-1 fallback in any of 12 conditions
[PASS] technical failures remain technical failures
[PASS] raw generations retained
[PASS] token-ceiling status retained
[PASS] identical corrected evaluator across Run-1 and Run-2
[PASS] empty-empty FDI/ICD does not receive F1=1
[PASS] no-RAG retrieval-copy rate is N/A


# Interpretation rules

1. **Do not treat technical failure as clinical abstention.**
2. **Run-1 and Run-2 use the same corrected evaluator.** The old erroneous Run-1 scoring is not reproduced.
3. `Retrieval_copy_rate` is `NaN` for no-RAG conditions because there is no retrieved evidence.
4. For FDI/ICD macro F1, empty-empty cases are excluded and the evaluable denominator is reported.
5. The clean reranking comparison is always **After − Before while generator and decoding are fixed**.
6. The new QLoRA adapter is trained once and reused unchanged across all QLoRA conditions.
7. Because the original training-time Exp2D exemplars are unavailable, the new adapter uses query-safe TF-IDF historical context. This must be disclosed as a new adaptation protocol rather than described as an exact recreation of the old adapter.
8. All generation JSON files are checkpointed after every patient, so interrupted runs can resume.


In [29]:
# ============================================================
# FINAL CELL — ZIP + DOWNLOAD ALL EXPERIMENT OUTPUTS
# Google Colab, outputs stored in /kaggle/working/
# ============================================================

import os
import shutil
import zipfile
from pathlib import Path
from google.colab import files

OUTPUT_DIR = Path("/kaggle/working")
ZIP_BASE = "/content/FINAL_12_CONDITION_ALL_OUTPUTS"
ZIP_PATH = ZIP_BASE + ".zip"

assert OUTPUT_DIR.exists(), f"Output directory not found: {OUTPUT_DIR}"

# ------------------------------------------------------------
# 1. Show everything that will be downloaded
# ------------------------------------------------------------

all_files = [
    p for p in OUTPUT_DIR.rglob("*")
    if p.is_file()
]

print("=" * 75)
print("EXPERIMENT OUTPUTS")
print("=" * 75)

total_size = 0

for p in sorted(all_files):
    size = p.stat().st_size
    total_size += size

    print(
        f"{size / (1024**2):9.2f} MB   "
        f"{p.relative_to(OUTPUT_DIR)}"
    )

print("\nFiles:", len(all_files))
print(
    f"Total uncompressed size: "
    f"{total_size / (1024**2):.2f} MB"
)


# ------------------------------------------------------------
# 2. Remove old ZIP if cell is re-run
# ------------------------------------------------------------

if os.path.exists(ZIP_PATH):
    os.remove(ZIP_PATH)


# ------------------------------------------------------------
# 3. ZIP entire /kaggle/working directory
# ------------------------------------------------------------

print("\nCreating ZIP...")

shutil.make_archive(
    ZIP_BASE,
    "zip",
    root_dir=str(OUTPUT_DIR)
)

assert os.path.exists(ZIP_PATH)

zip_size = os.path.getsize(ZIP_PATH)

print(
    f"✓ ZIP created: {ZIP_PATH}"
)
print(
    f"ZIP size: {zip_size / (1024**2):.2f} MB"
)


# ------------------------------------------------------------
# 4. Verify ZIP integrity
# ------------------------------------------------------------

print("\nVerifying ZIP...")

with zipfile.ZipFile(ZIP_PATH, "r") as z:

    bad_file = z.testzip()

    if bad_file is not None:
        raise RuntimeError(
            f"Corrupted file detected in ZIP: {bad_file}"
        )

    zip_contents = z.namelist()

print(
    f"✓ ZIP integrity passed "
    f"({len(zip_contents)} entries)"
)


# ------------------------------------------------------------
# 5. DOWNLOAD TO YOUR COMPUTER
# ------------------------------------------------------------

print("\nStarting download...")
print("Do not close this Colab tab until download begins.")

files.download(ZIP_PATH)

EXPERIMENT OUTPUTS
     0.00 MB   final_12_condition_llm_study/COMMON_PATIENT_FROZEN_AFTER.csv
     0.00 MB   final_12_condition_llm_study/COMMON_PATIENT_FROZEN_BEFORE.csv
     0.00 MB   final_12_condition_llm_study/COMMON_PATIENT_FROZEN_NONE.csv
     0.20 MB   final_12_condition_llm_study/COMMON_PATIENT_QLORA_AFTER.csv
     0.18 MB   final_12_condition_llm_study/COMMON_PATIENT_QLORA_BEFORE.csv
     0.22 MB   final_12_condition_llm_study/COMMON_PATIENT_QLORA_NONE.csv
     0.00 MB   final_12_condition_llm_study/CONTROLLED_DELTAS.csv
     0.00 MB   final_12_condition_llm_study/FINAL_12_CONDITION_RESULTS.csv
     0.01 MB   final_12_condition_llm_study/FINAL_12_CONDITION_RESULTS.json
     0.19 MB   final_12_condition_llm_study/FROZEN_AFTER_K5_RUN1.json
     0.17 MB   final_12_condition_llm_study/FROZEN_AFTER_K5_RUN2.json
     0.19 MB   final_12_condition_llm_study/FROZEN_BEFORE_K5_RUN1.json
     0.17 MB   final_12_condition_llm_study/FROZEN_BEFORE_K5_RUN2.json
     0.25 MB   final_12_condi

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [34]:
# ============================================================
# PAPER TABLE:
# No Retrieval vs Before Reranking vs After Reranking
# QLoRA MedGemma — Initial decoding configuration (Run1)
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. Extract the three directly comparable conditions
# ------------------------------------------------------------

paper_conditions = [
    "QLoRA | None | Run1",
    "QLoRA | Before | Run1",
    "QLoRA | After | Run1",
]

paper_df = final.loc[paper_conditions].copy()

# ------------------------------------------------------------
# 2. Keep only the metrics useful for the main paper
# ------------------------------------------------------------

paper_df = paper_df[
    [
        "Technical_success",
        "ROUGE_L",
        "METEOR",
        "FDI_macro_F1",
        "Unsupported_entity_patient_mean",
        "Retrieval_copy_rate",
        "Fabricated_tooth_rate",
    ]
].copy()

# ------------------------------------------------------------
# 3. Rename rows so notebook terminology does NOT appear
#    in the paper
# ------------------------------------------------------------

paper_df.index = [
    "No Retrieval",
    "Before Reranking",
    "After Reranking",
]

# ------------------------------------------------------------
# 4. Rename metrics to publication-friendly names
# ------------------------------------------------------------

paper_df = paper_df.rename(
    columns={
        "Technical_success": "Technical Success",
        "ROUGE_L": "ROUGE-L",
        "METEOR": "METEOR",
        "FDI_macro_F1": "FDI F1",
        "Unsupported_entity_patient_mean": "Unsupported Entity Rate",
        "Retrieval_copy_rate": "Retrieval Copy Rate",
        "Fabricated_tooth_rate": "Fabricated Tooth Rate",
    }
)

paper_df.index.name = "Generation Setting"

# ------------------------------------------------------------
# 5. Round for display
# ------------------------------------------------------------

paper_display = paper_df.round(3)

print("=" * 100)
print("QLoRA MedGemma — Initial decoding")
print("=" * 100)

display(paper_display)


QLoRA MedGemma — Initial decoding


,Technical Success,ROUGE-L,METEOR,FDI F1,Unsupported Entity Rate,Retrieval Copy Rate,Fabricated Tooth Rate
Generation Setting,,,,,,,
No Retrieval,0.990,0.062,0.043,0.000,1.000,NaN,1.000
Before Reranking,0.713,0.174,0.155,0.073,0.841,0.447,0.116
After Reranking,0.772,0.174,0.155,0.083,0.797,0.481,0.039


In [33]:
# ============================================================
# DECODING ABLATION:
# Initial decoding vs robustness-oriented decoding
# ============================================================

rows = []

for retrieval in ["Before", "After"]:

    run1 = final.loc[f"QLoRA | {retrieval} | Run1"]
    run2 = final.loc[f"QLoRA | {retrieval} | Run2"]

    for decoding_name, r in [
        ("Initial", run1),
        ("Robustness-oriented", run2),
    ]:
        rows.append({
            "Retrieval": (
                "Before Reranking"
                if retrieval == "Before"
                else "After Reranking"
            ),
            "Decoding": decoding_name,
            "Technical Success": r["Technical_success"],
            "ROUGE-L": r["ROUGE_L"],
            "FDI F1": r["FDI_macro_F1"],
            "Unsupported Entity Rate":
                r["Unsupported_entity_patient_mean"],
            "Retrieval Copy Rate":
                r["Retrieval_copy_rate"],
            "Fabricated Tooth Rate":
                r["Fabricated_tooth_rate"],
        })

decoding_table = pd.DataFrame(rows)

display(decoding_table.round(3))

,Retrieval,Decoding,Technical Success,ROUGE-L,FDI F1,Unsupported Entity Rate,Retrieval Copy Rate,Fabricated Tooth Rate
0,Before Reranking,Initial,0.713,0.174,0.073,0.841,0.447,0.116
1,Before Reranking,Robustness-oriented,0.762,0.140,0.012,0.968,0.045,0.183
2,After Reranking,Initial,0.772,0.174,0.083,0.797,0.481,0.039
3,After Reranking,Robustness-oriented,0.842,0.144,0.022,0.939,0.064,0.083
